# Занятие 5, демо 1. «Какая целевая функция лучше» - вопрос без ответа

Сравним две конструкции на **одних и тех же** данных, **одной и той же** сети и
с одинаковым бюджетом обучения:

- прямая интерполяция занятия 4: $x_\tau=(1-\tau)x_0+\tau x_1$, таргет $x_1-x_0$;
- гауссов путь занятия 5: $x_\tau=\alpha_\tau x_0+\sigma_\tau\epsilon$, таргет
  $v^{\mathrm{VP}}=\alpha_\tau\epsilon-\sigma_\tau x_0$.

Взято пять пар: внутри пары обе конструкции получают одну инициализацию и одни и
те же последовательности данных, шума и времени. Одной пары мало - на ней легко
увидеть то, чего нет.

In [ ]:
"""Общая база демо занятий 4 и 5: данные, сеть, метрика, солверы.

Здесь намеренно нет ни одной конструкции пути. На занятии 4 время идёт от шума
к данным, на занятии 5 - наоборот, и если держать обе ориентации в одном файле,
одно и то же имя начинает означать разное. Поэтому пути лежат по отдельности:
flow_cfm.py - занятие 4, flow_vp.py - занятие 5.

Направление интегрирования солверы получают аргументами t_from и t_to, а не
берут из умолчания: так его видно в месте вызова.

Датасет, сеть и расписание обучения те же, что в ДЗ-2, - иначе сравнение
занятий между собой перестало бы быть корректным.
"""
import math

import torch
from torch import nn

# --------------------------------------------------------------------------
# Данные: восемь гауссиан на окружности
# --------------------------------------------------------------------------

N_MODES = 8
RING_RADIUS = 2.0
MODE_STD = 0.15


def mode_centers(dtype=torch.float32) -> torch.Tensor:
    """Центры восьми компонент, форма [8, 2]."""
    k = torch.arange(N_MODES, dtype=dtype)
    angle = math.pi * k / 4
    return RING_RADIUS * torch.stack([torch.cos(angle), torch.sin(angle)], dim=-1)


def sample_data(n: int, generator: torch.Generator,
                dtype=torch.float32) -> torch.Tensor:
    """n точек из смеси восьми гауссиан, форма [n, 2]."""
    centers = mode_centers(dtype=dtype)
    which = torch.randint(N_MODES, (n,), generator=generator)
    noise = torch.randn(n, 2, generator=generator, dtype=dtype)
    return centers[which] + MODE_STD * noise


# --------------------------------------------------------------------------
# Сеть
# --------------------------------------------------------------------------

class VelocityNet(nn.Module):
    """Маленькая сеть: вход (x, tau) -> вектор в R^2. 4546 параметров."""

    def __init__(self, width: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, width),
            nn.SiLU(),
            nn.Linear(width, width),
            nn.SiLU(),
            nn.Linear(width, 2),
        )

    def forward(self, x: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([x, tau], dim=-1))


def make_model(seed: int = 0, width: int = 64) -> VelocityNet:
    """Сеть с воспроизводимой инициализацией."""
    state = torch.get_rng_state()
    try:
        torch.manual_seed(seed)
        model = VelocityNet(width=width)
    finally:
        torch.set_rng_state(state)
    return model


# --------------------------------------------------------------------------
# Метрика: energy distance
# --------------------------------------------------------------------------

def energy_distance(x: torch.Tensor, y: torch.Tensor) -> float:
    """Оценка energy distance между двумя выборками. Считается в float64."""
    x = x.detach().to(torch.float64)
    y = y.detach().to(torch.float64)
    n, m = x.shape[0], y.shape[0]
    cross = torch.cdist(x, y).sum() / (n * m)
    inner_x = torch.cdist(x, x).sum() / (n * n)
    inner_y = torch.cdist(y, y).sum() / (m * m)
    return float(2 * cross - inner_x - inner_y)


# --------------------------------------------------------------------------
# Солверы и счётчик вызовов
# --------------------------------------------------------------------------

class CountingField:
    """Обёртка над полем, считающая вызовы."""

    def __init__(self, field):
        self.field = field
        self.calls = 0

    def __call__(self, x: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        self.calls += 1
        return self.field(x, tau)


def euler_path(field, x: torch.Tensor, t_from: float, t_to: float,
               steps: int) -> torch.Tensor:
    """Явный Эйлер из t_from в t_to. Один шаг - один вызов поля.

    Возвращает все точки, форма [steps + 1, n, d].
    """
    if not isinstance(steps, int) or isinstance(steps, bool) or steps <= 0:
        raise ValueError("steps должно быть положительным int")

    h = (t_to - t_from) / steps
    ones = torch.ones(x.shape[0], 1, dtype=x.dtype, device=x.device)
    points = [x.clone()]

    with torch.no_grad():
        for m in range(steps):
            x = x + h * field(x, ones * (t_from + m * h))
            points.append(x.clone())

    return torch.stack(points)


def rk4_path(field, x: torch.Tensor, t_from: float, t_to: float,
             steps: int) -> torch.Tensor:
    """Классический RK4 из t_from в t_to. Один шаг - четыре вызова поля.

    Возвращает точки на границах макрошагов, форма [steps + 1, n, d].
    """
    if not isinstance(steps, int) or isinstance(steps, bool) or steps <= 0:
        raise ValueError("steps должно быть положительным int")

    h = (t_to - t_from) / steps
    ones = torch.ones(x.shape[0], 1, dtype=x.dtype, device=x.device)
    points = [x.clone()]

    with torch.no_grad():
        for m in range(steps):
            t = t_from + m * h
            k1 = field(x, ones * t)
            k2 = field(x + 0.5 * h * k1, ones * (t + 0.5 * h))
            k3 = field(x + 0.5 * h * k2, ones * (t + 0.5 * h))
            k4 = field(x + h * k3, ones * (t + h))
            x = x + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
            points.append(x.clone())

    return torch.stack(points)


def euler_solve(field, x: torch.Tensor, t_from: float, t_to: float,
                nfe: int) -> torch.Tensor:
    """Эйлер с бюджетом nfe вызовов поля: это ровно nfe шагов."""
    return euler_path(field, x, t_from, t_to, nfe)[-1]


def rk4_solve(field, x: torch.Tensor, t_from: float, t_to: float,
              nfe: int) -> torch.Tensor:
    """RK4 с бюджетом nfe вызовов поля: это nfe // 4 макрошагов."""
    if not isinstance(nfe, int) or isinstance(nfe, bool) or nfe <= 0:
        raise ValueError("nfe должно быть положительным int")
    if nfe % 4:
        raise ValueError("для RK4 бюджет вызовов должен делиться на 4")
    return rk4_path(field, x, t_from, t_to, nfe // 4)[-1]


# --------------------------------------------------------------------------
# Обучающий драйвер
# --------------------------------------------------------------------------

TRAIN_CONFIG = {
    "batch_size": 512,
    "steps": 12000,
    "lr": 2e-3,
    "data_seed": 1234,
    "noise_seed": 5678,
    "model_seed": 0,
    "report_every": 2000,
}


# --------------------------------------------------------------------------
# Замороженные веса: кладём вместе с тем, чем они являются
# --------------------------------------------------------------------------

def save_checkpoint(path, model, objective: str, cfg, history) -> None:
    """Сохраняет веса вместе с конструкцией, конфигом и достигнутыми потерями."""
    torch.save({"state_dict": model.state_dict(),
                "objective": objective,
                "config": dict(cfg),
                "final_loss": sum(history[-500:]) / 500}, path)


def load_checkpoint(path, objective: str):
    """Загружает веса и проверяет, что это обещанная конструкция.

    Демо занятий 4 и 5 стоят на утверждении «одна архитектура, один бюджет
    обучения». Утверждение, которое нельзя проверить на месте, - это дыра:
    файл легко перепутать, и ошибка будет молчаливой. Поэтому конструкция
    лежит внутри файла и сверяется при загрузке.
    """
    blob = torch.load(path, map_location="cpu", weights_only=False)
    if blob["objective"] != objective:
        raise ValueError(f"{path}: обучено на '{blob['objective']}', "
                         f"ожидалось '{objective}'")
    model = make_model(seed=blob["config"]["model_seed"])
    model.load_state_dict(blob["state_dict"])
    model.eval()
    return model, blob


"""Занятие 4: прямая интерполяция и conditional flow matching.

Ориентация занятия 4: `noise` - источник, `data` - цель, tau идёт от 0 к 1,
генерация вперёд. Имя x_0 здесь не используется намеренно: на занятии 5 оно
означает ровно противоположное.

Требует flow_common.py.
"""
import torch



def straight_interpolation(noise, data, tau):
    """Прямой отрезок между источником и целью."""
    return (1.0 - tau) * noise + tau * data


def cfm_target(noise, data):
    """Производная вдоль отрезка: она постоянна и равна разности концов."""
    return data - noise


def cfm_loss(model, noise, data, tau):
    """Conditional flow matching на прямой интерполяции."""
    prediction = model(straight_interpolation(noise, data, tau), tau)
    return ((prediction - cfm_target(noise, data)) ** 2).sum(dim=-1).mean()


def train_cfm(cfg=None, verbose=True):
    """Обучение поля на прямой интерполяции. Возвращает (model, история)."""
    cfg = dict(TRAIN_CONFIG if cfg is None else cfg)
    model = make_model(seed=cfg["model_seed"])
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    data_gen = torch.Generator().manual_seed(cfg["data_seed"])
    noise_gen = torch.Generator().manual_seed(cfg["noise_seed"])
    batch = cfg["batch_size"]
    history = []

    for step in range(1, cfg["steps"] + 1):
        data = sample_data(batch, data_gen)
        noise = torch.randn(batch, 2, generator=noise_gen)
        tau = torch.rand(batch, 1, generator=noise_gen)

        loss = cfm_loss(model, noise, data, tau)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        history.append(loss.detach().item())

        if verbose and step % cfg["report_every"] == 0:
            window = history[-cfg["report_every"]:]
            print(f"  шаг {step:6d}   потери {sum(window) / len(window):.4f}")

    return model, history


def learned_field(model):
    """Поле ОДУ занятия 4: предсказание сети как есть, без множителей."""
    def field(x, tau):
        return model(x, tau)
    return field


def oracle_field(x, tau):
    """Точное population-optimal поле E[data - noise | x_tau = x].

    Нейросети здесь нет: для нашей смеси гауссиан условное среднее выписывается
    аналитически. Пусть a = 1 - tau, b = tau, s = MODE_STD, D = a^2 + b^2 s^2.
    Тогда веса компонент пропорциональны exp(-||x - b mu_k||^2 / 2D), а внутри
    компоненты условное среднее равно mu_k + (b s^2 - a) / D * (x - b mu_k).

    Нужно, чтобы отделить свойство конструкции от ошибки обучения: если кривые
    траектории видны и здесь, дело не в том, что сеть маленькая или недоучена.
    """
    a = 1.0 - tau                                          # [n, 1]
    b = tau                                                # [n, 1]
    mu = mode_centers(dtype=x.dtype)                       # [8, 2]
    var = a * a + b * b * MODE_STD ** 2                    # [n, 1]
    shift = x.unsqueeze(1) - b.unsqueeze(1) * mu           # [n, 8, 2]
    weights = torch.softmax(-(shift ** 2).sum(-1) / (2 * var), dim=-1)
    coefficient = ((b * MODE_STD ** 2 - a) / var).unsqueeze(1)
    per_mode = mu.unsqueeze(0) + coefficient * shift       # [n, 8, 2]
    return (weights.unsqueeze(-1) * per_mode).sum(dim=1)


def path_ratio(points):
    """Длина пути, делённая на длину хорды. Единица - прямая."""
    length = (points[1:] - points[:-1]).norm(dim=-1).sum(dim=0)
    chord = (points[-1] - points[0]).norm(dim=-1)
    return length / chord


def cfm_sample(model, noise, nfe):
    """Генерация полем занятия 4: Эйлер из tau=0 в tau=1."""
    return euler_solve(learned_field(model), noise, 0.0, 1.0, nfe)


"""Занятие 5: гауссов угловой путь.

Ориентация занятия 5 (та же, что в ДЗ-2): x_0 - это **данные**, eps - шум,
tau идёт от 0 (данные) к 1 (чистый шум), а генерация - в обратную сторону,
от 1 к 0. На занятии 4 время идёт наоборот, поэтому пути и лежат в разных
файлах.

Требует flow_common.py.
"""
import math

import torch



def angular_schedule(tau: torch.Tensor):
    """Возвращает (alpha_tau, sigma_tau): alpha = cos(pi*tau/2), sigma = sin(...).

    При tau=0 это чистые данные, при tau=1 - чистый шум.
    """
    phi = (math.pi / 2) * tau
    return torch.cos(phi), torch.sin(phi)


def vp_target(x_0: torch.Tensor, eps: torch.Tensor,
              tau: torch.Tensor) -> torch.Tensor:
    """Условный таргет v^VP = alpha_tau * eps - sigma_tau * x_0."""
    alpha, sigma = angular_schedule(tau)
    return alpha * eps - sigma * x_0


def vp_loss(model, x_0: torch.Tensor, eps: torch.Tensor,
            tau: torch.Tensor) -> torch.Tensor:
    """Средний квадрат нормы ошибки предсказания v^VP."""
    alpha, sigma = angular_schedule(tau)
    x_tau = alpha * x_0 + sigma * eps
    return ((model(x_tau, tau) - vp_target(x_0, eps, tau)) ** 2).sum(dim=-1).mean()


def train_vp(cfg=None, verbose=True):
    """Обучение поля на гауссовом пути. Возвращает (model, история)."""
    cfg = dict(TRAIN_CONFIG if cfg is None else cfg)
    model = make_model(seed=cfg["model_seed"])
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    data_gen = torch.Generator().manual_seed(cfg["data_seed"])
    noise_gen = torch.Generator().manual_seed(cfg["noise_seed"])
    batch = cfg["batch_size"]
    history = []

    for step in range(1, cfg["steps"] + 1):
        x_0 = sample_data(batch, data_gen)
        eps = torch.randn(batch, 2, generator=noise_gen)
        tau = torch.rand(batch, 1, generator=noise_gen)

        loss = vp_loss(model, x_0, eps, tau)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        history.append(loss.detach().item())

        if verbose and step % cfg["report_every"] == 0:
            window = history[-cfg["report_every"]:]
            print(f"  шаг {step:6d}   потери {sum(window) / len(window):.4f}")

    return model, history


def learned_vp_field(model):
    """Поле ОДУ занятия 5: U_tau = (pi/2) * v^VP."""
    def field(x, tau):
        return (math.pi / 2) * model(x, tau)
    return field


def vp_sample(model, z_1, nfe):
    """Генерация полем занятия 5: Эйлер из tau=1 в tau=0."""
    return euler_solve(learned_vp_field(model), z_1, 1.0, 0.0, nfe)

In [ ]:
pairs = [(load_checkpoint(f"cfm_seed{s}.pt", "cfm")[0],
          load_checkpoint(f"vp_seed{s}.pt", "vp")[0]) for s in range(5)]

_, blob = load_checkpoint("cfm_seed0.pt", "cfm")
print(f"пар: {len(pairs)}, шагов обучения {blob['config']['steps']}, "
      f"параметров {sum(p.numel() for p in pairs[0][0].parameters())}")

## Качество по числу вызовов сети

По пять пар обучения и по три набора шума на каждую: пятнадцать парных сравнений
на каждый бюджет. Печатаем медианы energy distance и то, сколько раз из
пятнадцати выиграл прямой путь.

In [ ]:
from statistics import median

reference = sample_data(2048, torch.Generator().manual_seed(99))
noises = [torch.randn(2048, 2, generator=torch.Generator().manual_seed(s))
          for s in (7, 21, 35)]

floor = median([energy_distance(sample_data(2048, torch.Generator().manual_seed(1000 + 2 * s)),
                                sample_data(2048, torch.Generator().manual_seed(1001 + 2 * s)))
                for s in range(5)])
print(f"пол метрики (данные против данных): {floor:.4f}")
print()

print(f"{'NFE':>5}  {'прямой':>9}  {'гауссов':>9}  {'прямой выиграл':>16}")
for nfe in (2, 4, 8, 16, 32):
    straight, gaussian = [], []
    for cfm, vp in pairs:
        for noise in noises:
            straight.append(energy_distance(cfm_sample(cfm, noise, nfe), reference))
            gaussian.append(energy_distance(vp_sample(vp, noise, nfe), reference))
    wins = sum(a < b for a, b in zip(straight, gaussian))
    print(f"{nfe:>5}  {median(straight):>9.4f}  {median(gaussian):>9.4f}"
          f"  {wins:>13} / {len(straight)}")

## Что здесь на самом деле видно

Порядок **не** меняется: на этом датасете и по этой метрике гауссов путь лучше
почти во всех пятнадцати сравнениях. Меняется размер разницы. На двух вызовах
прямой путь хуже примерно **в восемь раз**, на тридцати двух - процентов на
десять. И этот остаток уже вчетверо меньше, чем пол самой метрики в первой
строке: различить конструкции она там просто не может. Одно и то же сравнение при
одном бюджете читается как «конструкции неразличимы», при другом - как «одна
провалилась».

Поэтому число качества без указания $N_{\mathrm{FE}}$ не значит ничего. И это ещё
при фиксированных датасете и метрике: сменится любое из двух - и сравнивать
придётся заново.

Кстати, первая строка таблицы объясняет, почему в статьях так любят малые
бюджеты: именно там между конструкциями видна разница.

Отдельно стоит сказать, чего этот опыт **не** показывает. Здесь различаются не
«целевые функции», а две полные конструкции целиком: путь и вместе с ним
распределение точек $x_\tau$, масштаб таргета ($\mathbb E\lVert x_1-x_0\rVert^2
\approx 6.05$ против $\mathbb E\lVert v^{\mathrm{VP}}\rVert^2\approx 3.02$) и
сложность самой регрессии. А вот выбор времени и веса в лоссе у них как раз
одинаковые: $\tau\sim U[0,1]$ и невзвешенный MSE в обоих случаях.

## Веса по времени: где они появляются на самом деле

Чтобы увидеть неявные веса, менять путь не нужно - достаточно менять, **что
именно** сеть называет своим предсказанием. На гауссовом пути три величины
связаны тождествами

$$
\epsilon=\sigma_\tau x_\tau+\alpha_\tau v,\qquad
x_0=\alpha_\tau x_\tau-\sigma_\tau v
$$

Значит из одного и того же предсказания $\hat v$ бесплатно получаются $\hat
\epsilon$ и $\hat x_0$. Ошибки при этом связаны жёстко:

$$
\hat\epsilon-\epsilon=\alpha_\tau(\hat v-v),\qquad
\hat x_0-x_0=-\sigma_\tau(\hat v-v)
$$

Возьмём **одну** обученную сеть и посчитаем три «разные целевые функции» на
одном и том же батче.

In [ ]:
vp, _ = load_checkpoint("vp_seed0.pt", "vp")

g = torch.Generator().manual_seed(2024)
x_0 = sample_data(20_000, g)
eps = torch.randn(20_000, 2, generator=g)
tau = torch.rand(20_000, 1, generator=g)

alpha, sigma = angular_schedule(tau)
x_tau = alpha * x_0 + sigma * eps
v = vp_target(x_0, eps, tau)
with torch.no_grad():
    v_hat = vp(x_tau, tau)

mse = lambda a, b: float(((a - b) ** 2).sum(-1).mean())
squared = ((v_hat - v) ** 2).sum(-1, keepdim=True)

print(f"MSE по v     {mse(v_hat, v):>8.4f}")
print(f"MSE по eps   {mse(sigma * x_tau + alpha * v_hat, eps):>8.4f}"
      f"   = MSE по v с весом alpha^2: {float((alpha ** 2 * squared).mean()):.4f}")
print(f"MSE по x_0   {mse(alpha * x_tau - sigma * v_hat, x_0):>8.4f}"
      f"   = MSE по v с весом sigma^2: {float((sigma ** 2 * squared).mean()):.4f}")
print()
print("alpha^2 + sigma^2 = 1, поэтому две нижние строки складываются в верхнюю")

## Что из этого следует

Сеть одна, путь один, батч один - а «значение целевой функции» получилось три
разных. Выбор предсказываемой величины **и есть** выбор веса по времени:
$\epsilon$-параметризация даёт вес $\alpha_\tau^2$ и почти игнорирует
зашумлённый конец, $x_0$-параметризация даёт зеркальный $\sigma_\tau^2$ и
игнорирует чистый. Третьего свободного параметра тут нет: веса складываются в
единицу.

Отсюда ответ на вопрос «почему в статьях сравнивают столько вариантов». Строка
«наш лосс лучше» без указания пути, параметризации, весов, метрики и
$N_{\mathrm{FE}}$ не сообщает ничего: каждая из пяти вещей двигает число сама по
себе. Сравнение имеет смысл только тогда, когда меняется ровно одна из них.